# NCS 기본곡 50곡 채보 — GPU 서버 실행 (S15P31A307-322)

NCS 50곡을 서비스 채보 명령(`ssaplash-chart run`: 드럼 분리 → 채보)으로 처리해 곡별 채보 JSON과 드럼리스 MR을 만든다. **위에서부터 차례로 실행한다.**

**코드는 서비스 워커 폴더를 그대로 씀.** `ssaplash/S15P31A307/AI`(워커가 쓰는 체크아웃)의 코드·가중치·`.venv`로 돌려 서비스와 같은 결과를 낸다. 이 폴더는 읽기만 하고 `git pull`이나 파일 수정은 하지 않는다. 채보 모델 이름은 `ssaplash/worker.env`의 `SSAPLASH_CHART_MODEL` 한 줄만 읽어 쓴다(토큰 등 다른 값은 읽지 않음).

**준비물.** `ncs_audio.zip`을 `ssaplash/upload/`에, 이 노트북을 `ssaplash/notebooks/`에 올린다.

**순서.** 1 설정 → 2 음원 풀기 → 3 50곡 시작(백그라운드) → 4 진행 확인(여러 번) → 5 결과 묶기.

**끊기면.** 3번은 nohup이라 브라우저를 닫아도 돈다. 커널이 다시 시작됐으면 1번 다음 4번으로 본다. 도중에 멈췄으면 1 → 3을 다시 친다. 이미 `chart.json`이 있는 곡은 건너뛰고 남은 곡만 돈다.

**결과.** 곡 파일 이름이 `ncs_01 Cartoon, Jéja - On & On (feat. Daniel Levi).opus`처럼 번호 + 아티스트 - 제목이고, 결과 폴더도 같은 이름이다. `ssaplash/ncs/charts/<곡 이름>/`에 `chart.json`·`transcription.json`·`separation.json`·`accompaniment.flac`(드럼리스 MR)·`drums.flac`. 5번 zip에는 드럼 트랙(`drums.flac`)을 뺀 네 파일만 담는다.

In [ ]:
# 1. 설정. 지정받은 GPU는 6번 (여러 번 쳐도 됨)
import os
import subprocess
import sys
from pathlib import Path

GPU_ID = "6"
HOME = Path.home()
WORK = HOME / "ssaplash"
MAIN = WORK / "S15P31A307" / "AI"          # 서비스 워커 코드 폴더. 읽기만 함
CHART_BIN = MAIN / ".venv" / "bin" / "ssaplash-chart"
UPLOAD = WORK / "upload"
NCS = WORK / "ncs"
AUDIO = NCS / "audio"
CHARTS = NCS / "charts"
RUNNER = NCS / "run_ncs.sh"
LOG = NCS / "run.log"
PID = NCS / "run.pid"

# worker.env에서 채보 모델 이름 한 줄만 읽음
chart_model = ""
for line in (WORK / "worker.env").read_text().splitlines():
    line = line.strip().removeprefix("export ")
    if line.startswith("SSAPLASH_CHART_MODEL="):
        chart_model = line.split("=", 1)[1].split("#")[0].strip()
print("채보 모델:", chart_model or "(worker.env에 없음 → 설정 파일 기본값)")
!nvidia-smi -i $GPU_ID --query-gpu=index,name,memory.used,memory.total --format=csv
!df -h $HOME | tail -1
!git -C $MAIN log -1 --format='%h %ci %s'
!ls -la $CHART_BIN $UPLOAD/ncs_audio.zip
# 확인: 채보 모델 이름 한 줄(워커 설정과 같은 값, 예 strum),
#       GPU 한 줄(남은 메모리 20GB 이상), 디스크 여유 5GB 이상,
#       워커 코드 커밋 한 줄(이 커밋이 서비스 코드. 나중에 결과 기록에 씀),
#       ssaplash-chart와 ncs_audio.zip 두 파일이 찍힘(zip은 오늘 날짜, 약 174MB)

In [ ]:
# 2. 음원 풀기 (여러 번 쳐도 됨). 예전에 푼 곡 파일은 지우고 새로 풂
NCS.mkdir(parents=True, exist_ok=True)
for old in AUDIO.glob("ncs_*.opus"):
    old.unlink()
!{sys.executable} -m zipfile -e $UPLOAD/ncs_audio.zip $NCS
opus = sorted(AUDIO.glob("ncs_*.opus"))
print(len(opus), "곡")
print(opus[0].name)
print(opus[-1].name)
# 확인: '50 곡', 'ncs_01 Cartoon, Jéja - On & On (feat. Daniel Levi).opus', 'ncs_50 Netrum, Halvorsen - Phoenix.opus'

In [ ]:
# 3. 50곡 시작 (백그라운드). 이미 돌고 있으면 새로 띄우지 않음
script = f'''#!/bin/bash
# 곡마다 서비스 채보 명령 실행. chart.json이 이미 있는 곡은 건너뜀
export CUDA_VISIBLE_DEVICES={GPU_ID}
export PYTHONUTF8=1
# 노트북 커널의 그래프 설정(matplotlib_inline)을 물려받으면 분리 모델 올리기가 실패함. 화면 없는 설정으로 고정
export MPLBACKEND=Agg
export OMP_NUM_THREADS=4 OPENBLAS_NUM_THREADS=4 MKL_NUM_THREADS=4
{"export SSAPLASH_CHART_MODEL=" + chart_model if chart_model else "unset SSAPLASH_CHART_MODEL"}
cd {MAIN}
for f in {AUDIO}/ncs_*.opus; do
  n=$(basename "$f" .opus)
  out={CHARTS}/$n
  if [ -f "$out/chart.json" ]; then echo "$n 건너뜀 (이미 있음)"; continue; fi
  mkdir -p "$out"
  start=$(date +%s)
  if {CHART_BIN} run "$f" --out "$out" --device cuda > "$out/run.log" 2>&1; then
    echo "$n 끝 $(( $(date +%s) - start ))초"
  else
    echo "$n 실패 (로그 $out/run.log)"
  fi
done
echo "전체 끝 $(date '+%F %T')"
'''
RUNNER.write_text(script)
RUNNER.chmod(0o755)
running = PID.exists() and subprocess.run(["kill", "-0", PID.read_text().strip()], capture_output=True).returncode == 0
if running:
    print("이미 실행 중, pid", PID.read_text().strip())
else:
    CHARTS.mkdir(parents=True, exist_ok=True)
    # 로그는 새로 씀. 지난 실행의 실패 줄이 4번 셀에 다시 잡히지 않게
    p = subprocess.Popen(f"nohup {RUNNER} > {LOG} 2>&1 & echo $!", shell=True, stdout=subprocess.PIPE, text=True)
    PID.write_text(p.stdout.read().strip())
    print("시작, pid", PID.read_text().strip(), "로그", LOG)
# 확인: '시작, pid 숫자 로그 .../ncs/run.log'. 두 번째부터는 '이미 실행 중, pid 숫자'
# 처음 곡은 모델을 올리느라 더 걸림. 곡마다 따로 실행해서 곡당 모델 올리는 시간이 들어감

In [ ]:
# 4. 진행 확인 (여러 번)
stems = {p.stem for p in AUDIO.glob("ncs_*.opus")}
done = sorted(p.parent.name for p in CHARTS.glob("ncs_*/chart.json") if p.parent.name in stems)
print(f"{len(done)}/50곡 끝")
!tail -n 8 $LOG
running = PID.exists() and subprocess.run(["kill", "-0", PID.read_text().strip()], capture_output=True).returncode == 0
print("실행 중" if running else "멈춤")
failed = [l for l in LOG.read_text().splitlines() if "실패" in l] if LOG.exists() else []
print("실패:", failed or "없음")
for line in failed[:3]:
    n = line.rsplit(" 실패", 1)[0]
    print("----", n)
    print("".join((CHARTS / n / "run.log").read_text().splitlines(keepends=True)[-15:]))
# 확인: 'N/50곡 끝'이 늘어남, 로그에 'ncs_NN 끝 초' 줄. 다 끝나면 '50/50곡 끝', 로그 마지막 '전체 끝 날짜', '멈춤', '실패: 없음'
# 실패가 있으면 아래에 그 곡 run.log 끝 15줄이 찍힘. 그 내용을 채팅에 붙여 줌
# 멈춤인데 50곡이 안 됐으면 1 → 3을 다시 쳐서 남은 곡만 돌림

In [ ]:
# 5. 결과 묶기. 드럼 트랙은 빼고 곡마다 네 파일만
import zipfile
dest = UPLOAD / "ncs_charts.zip"
names = ["chart.json", "transcription.json", "separation.json", "accompaniment.flac"]
missing = []
stems = sorted(p.stem for p in AUDIO.glob("ncs_*.opus"))  # 지금 음원 이름과 같은 폴더만 담음
with zipfile.ZipFile(dest, "w", zipfile.ZIP_STORED) as z:
    for d in (CHARTS / s for s in stems):
        for name in names:
            f = d / name
            if f.is_file():
                z.write(f, f"charts/{d.name}/{name}")
            else:
                missing.append(f"{d.name}/{name}")
    z.write(LOG, "charts/run.log")
print(dest, f"{dest.stat().st_size / 2**20:.0f} MiB")
print("빠진 파일:", missing or "없음")
!git -C $MAIN log -1 --format='%h %ci %s'
# 확인: '.../upload/ncs_charts.zip 숫자 MiB'(1~2GB 안팎), '빠진 파일: 없음', 워커 코드 커밋 한 줄
# 이 zip을 내려받아 로컬 AI/data/ncs/에 풀면 AI/data/ncs/charts/ncs_NN/이 생김. 커밋 줄은 채팅에 붙여 줌